# Test Robot Motion

Runs MuJoCo and loads the MJCF file to simulate the robot.

In [1]:
# Import standard libraries
from pathlib import Path
import time

# Import third-party libraries
import mujoco
import mujoco.viewer

In [23]:
# Settings
MJCF_PATH = Path("/workspace/mechanical/FreeCAD/bala2-fire/bala2-fire-simplified.xml")
MOTOR_VEL_STEP = 0.1
MOTOR_VEL_LIMIT = 1.0

In [24]:
# Load model
model = mujoco.MjModel.from_xml_path(str(MJCF_PATH))
data  = mujoco.MjData(model)

In [25]:
STEP  = 0.1
LIMIT = 1.0

ctrl = {"left": 0.0, "right": 0.0}

KEY_BACKSPACE = 259
KEY_UP        = 265
KEY_DOWN      = 264

def clamp(x):
    return max(-LIMIT, min(LIMIT, x))

def key_callback(keycode):
    if keycode == KEY_BACKSPACE:
        # Viewer's built-in handler resets the sim state; we just zero
        # our Python-side ctrl so the motors don't torque it back up.
        ctrl["left"] = ctrl["right"] = 0.0
        print("reset")
        return

    if keycode == KEY_UP:
        ctrl["left"]  = clamp(ctrl["left"]  + STEP)
        ctrl["right"] = clamp(ctrl["right"] + STEP)
    elif keycode == KEY_DOWN:
        ctrl["left"]  = clamp(ctrl["left"]  - STEP)
        ctrl["right"] = clamp(ctrl["right"] - STEP)
    else:
        return

    print(f"left={ctrl['left']:+.2f}  right={ctrl['right']:+.2f}")


In [ ]:
mujoco.mj_resetData(model, data)

print("Controls: E/D = left motor +/-, I/K = right motor +/-, 0 = stop")
print("Backspace resets the sim. Close the viewer window to exit.\n")

with mujoco.viewer.launch_passive(model, data, key_callback=key_callback) as viewer:
    # Define free look camera
    viewer.cam.type = mujoco.mjtCamera.mjCAMERA_FREE
    viewer.cam.lookat[:] = [0, 0, 0.05]   # point at the robot's mid-height
    viewer.cam.distance  = 0.8             # meters from lookat
    viewer.cam.azimuth   = 45            # degrees; try values from -180 to 180
    viewer.cam.elevation = -25             # degrees; negative looks down

    # Simulation loop
    while viewer.is_running():
        step_start = time.time()

        # Write the latest ctrl values into the sim.
        # Index order matches the <actuator> block in the MJCF:
        #   0 = left_motor, 1 = right_motor.
        data.ctrl[0] = ctrl["left"]
        data.ctrl[1] = ctrl["right"]

        mujoco.mj_step(model, data)
        viewer.sync()

        slack = model.opt.timestep - (time.time() - step_start)
        if slack > 0:
            time.sleep(slack)

Controls: E/D = left motor +/-, I/K = right motor +/-, 0 = stop
Backspace resets the sim. Close the viewer window to exit.

left=-0.10  right=-0.10
reset
left=-0.10  right=-0.10
left=+0.00  right=+0.00
left=+0.10  right=+0.10
left=+0.20  right=+0.20
left=+0.30  right=+0.30
left=+0.40  right=+0.40
reset
left=-0.10  right=-0.10
left=+0.00  right=+0.00
left=+0.10  right=+0.10
left=+0.20  right=+0.20
left=+0.30  right=+0.30
left=+0.40  right=+0.40
reset
left=-0.10  right=-0.10
left=+0.00  right=+0.00
reset
left=-0.10  right=-0.10
left=+0.00  right=+0.00
left=+0.10  right=+0.10
left=+0.20  right=+0.20
left=+0.30  right=+0.30
left=+0.40  right=+0.40
left=+0.50  right=+0.50
left=+0.60  right=+0.60
left=+0.70  right=+0.70
left=+0.80  right=+0.80
reset
left=-0.10  right=-0.10
left=+0.00  right=+0.00
left=+0.10  right=+0.10
left=+0.20  right=+0.20
left=+0.30  right=+0.30
reset
left=-0.10  right=-0.10
left=+0.00  right=+0.00
left=+0.10  right=+0.10
left=+0.20  right=+0.20
left=+0.30  right=+0.30
